# Polarized sunglint

This tutorial explores the polarization of the sunglint computed by `coxmunk`,
following [Polarized sunglint](../polarization.rst) and Harmel & Chami (2013).
All the quantities are computed at the sea surface, without atmosphere.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from coxmunk import sunglint
from coxmunk.visu import plot
from coxmunk.wind import glint_directions

## Brewster angle

For an unpolarized incident light, the degree of linear polarization of the
reflected light only depends on the angle of incidence $\omega$ on the facet,
i.e., on the scattering angle $\Theta = \pi - 2\omega$. It reaches 1 at the
Brewster angle $\omega_B = \arctan m$.

In [ ]:
m = 1.334
omega = np.radians(np.linspace(0, 89.9, 500))
R = np.array([sunglint(0, 0, 0, m=m).fresnel(w)[:, 0] for w in omega])
dolp = np.abs(R[:, 1]) / R[:, 0]

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].plot(np.degrees(omega), R[:, 0], label='$R_{11}$ (reflectance)')
axs[0].plot(np.degrees(omega), -R[:, 1], label='$-R_{21}$ (polarized reflectance)')
axs[0].set_xlabel(r'angle of incidence $\omega$ (deg)')
axs[0].set_yscale('log')
axs[0].legend()
axs[1].plot(180 - 2 * np.degrees(omega), dolp)
axs[1].axvline(180 - 2 * np.degrees(np.arctan(m)), color='k', ls=':', label='Brewster')
axs[1].set_xlabel(r'scattering angle $\Theta$ (deg)')
axs[1].set_ylabel('degree of linear polarization')
axs[1].legend()
plt.tight_layout()
print(f'Brewster angle: {np.degrees(np.arctan(m)):.1f} deg, scattering angle: {180 - 2 * np.degrees(np.arctan(m)):.1f} deg')

## Stokes parameters of the sunglint

Polar maps of $I$, $Q$, $U$ for a Sun at 35°, a wind azimuth of 0 and wind
speeds of 2, 6 and 10 m s$^{-1}$, over viewing angles up to 70° (the
configuration of Fig. 1 of Harmel & Chami, 2013, here without atmosphere).

In [ ]:
sza = 35
vza = np.linspace(0, 70, 36)
azi = np.linspace(0, 360, 91)


def stokes_map(ws, wazi=0, stats='bh2006', m=1.334):
    S = np.empty((3, len(azi), len(vza)))
    for j, a in enumerate(azi):
        for i, v in enumerate(vza):
            S[:, j, i] = sunglint(sza, v, a, m=m).sunglint(ws, wazi, stats=stats)[:3]
    return S


winds = (2, 6, 10)
maps = {ws: stokes_map(ws) for ws in winds}

fig, axs = plt.subplots(3, 3, subplot_kw=dict(projection='polar'), figsize=(16, 15))
p = plot()
for k, ws in enumerate(winds):
    I, Q, U = maps[ws]
    p.add_polplot(axs[0, k], vza, azi, I.T, title=f'I, {ws} m/s', cmap=plt.cm.Spectral_r, vmin=0, vmax=0.5)
    p.add_polplot(axs[1, k], vza, azi, Q.T, title=f'Q, {ws} m/s', cmap=plt.cm.RdBu_r, vmin=-0.15, vmax=0.15)
    p.add_polplot(axs[2, k], vza, azi, U.T, title=f'U, {ws} m/s', cmap=plt.cm.RdBu_r, vmin=-0.05, vmax=0.05)
plt.tight_layout()

The fraction of the viewing directions in which the sunglint normalized
radiance $\mu_s I$ exceeds the PARASOL noise ($4\times10^{-4}$) grows with the
wind speed:

In [ ]:
for ws in winds:
    frac = np.mean(np.cos(np.radians(sza)) * maps[ws][0] > 4e-4)
    print(f'{ws:>2} m/s: {100 * frac:.0f} % of the directions affected by sunglint')

These fractions are computed at the sea surface, without atmosphere, on a
regular grid of viewing angles: they are not directly comparable with the
top-of-atmosphere values discussed in Harmel & Chami (2013).

## Polarization does not depend on the wind

The wind speed changes the amount of glint, through the slope density, but not
its degree of polarization, which only depends on the geometry. Where the glint
is not negligible, the degree of linear polarization is identical for all wind
speeds:

In [ ]:
dolp = {ws: np.hypot(maps[ws][1], maps[ws][2]) / maps[ws][0] for ws in winds}
both = (maps[2][0] > 1e-6) & (maps[10][0] > 1e-6)
print('max |DoLP(2 m/s) - DoLP(10 m/s)| =', np.nanmax(np.abs(dolp[2] - dolp[10])[both]))

fig, ax = plt.subplots(subplot_kw=dict(projection='polar'), figsize=(7, 7))
plot().add_polplot(ax, vza, azi, np.where(maps[10][0] > 1e-6, dolp[10], np.nan).T,
                   title='DoLP of the sunglint', cmap=plt.cm.viridis, colfmt='%0.2f')
plt.tight_layout()

## Spectral flatness

Between 670 and 865 nm, the refractive index of seawater varies from 1.337 to
1.331. The effect on the sunglint Stokes parameters is about 2 to 3 %, so that
the sunglint is nearly spectrally flat:

In [ ]:
S670 = stokes_map(6, m=1.337)
S865 = stokes_map(6, m=1.331)
ok = np.abs(S670) > 1e-3
for k, name in enumerate('IQU'):
    ratio = S865[k][ok[k]] / S670[k][ok[k]]
    print(f'{name}(865)/{name}(670): min {ratio.min():.3f}, max {ratio.max():.3f}')

## Directions affected by sunglint

PARASOL observes a ground target under up to 16 directions. With an ancillary
wind speed, {func}`coxmunk.wind.glint_directions` flags the directions that can
be affected by the glint (wind speed $\pm$ 1 m s$^{-1}$, PARASOL noise). In
Harmel & Chami (2013), a direction detected as contaminated by the POLAC-glint
algorithm but geometrically out of the glint is attributed to clouds.

In [ ]:
signed_vza = np.linspace(-60, 60, 16)
geometries = np.array([(sza, abs(v), 150 if v >= 0 else 330) for v in signed_vza])

fig, ax = plt.subplots(figsize=(9, 4.5))
for ws in winds:
    flags = glint_directions(geometries, ws)
    I = np.array([sunglint(*g).sunglint(ws, 0, stats='bh2006')[0] for g in geometries])
    line, = ax.semilogy(signed_vza, np.cos(np.radians(sza)) * np.clip(I, 1e-8, None), '-', label=f'{ws} m/s')
    ax.semilogy(signed_vza[flags], np.cos(np.radians(sza)) * np.clip(I[flags], 1e-8, None), 'o', color=line.get_color())
ax.axhline(4e-4, color='k', ls=':', label='PARASOL noise')
ax.set_ylim(1e-6, 1)
ax.set_xlabel('viewing zenith angle (deg, negative: backward)')
ax.set_ylabel(r'normalized radiance $\mu_s I$')
ax.set_title('PARASOL-like sequence (dots: flagged as glint)')
ax.legend()
plt.show()